# Notebook 09: Persistence — How Redis Saves Data to Disk

Redis stores data in **memory (RAM)**, which is why it's so fast. But what happens when the server restarts? Without persistence, all data would be lost!

Redis provides **two persistence mechanisms**:

1. **RDB (Redis Database)** — Point-in-time **snapshots** (like taking a photo)
2. **AOF (Append Only File)** — Logs **every write operation** (like keeping a diary)

You can use either, both, or neither (pure cache mode).

In [ ]:
import redis
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
print(f"Connected! Redis version: {r.info('server')['redis_version']}")

---
## 1. RDB Snapshots

RDB creates a **complete snapshot** of all data at a point in time and saves it to a `.rdb` file.

**How it works:**
1. Redis forks a child process
2. The child writes all data to a temporary file
3. When done, it replaces the old `.rdb` file
4. The main process continues serving requests uninterrupted

**Think of it like:** Taking a photo of your whiteboard every hour.

In [ ]:
# Check current RDB configuration
save_config = r.config_get('save')
print(f"RDB save intervals: {save_config}")
print("")
print("Format: 'seconds changes' — save if N changes in M seconds")
print("Example: '3600 1' means 'save if at least 1 change in 3600 seconds'")

# When was the last save?
last_save = r.lastsave()
print(f"\nLast RDB save: {last_save}")

In [ ]:
# Trigger a manual background save
# Redis CLI: BGSAVE
r.bgsave()
print("Background save triggered!")

time.sleep(1)  # Wait for save to complete

last_save = r.lastsave()
print(f"Last save timestamp: {last_save}")

# Check persistence info
info = r.info('persistence')
print(f"\nRDB info:")
print(f"  Last save status:  {info.get('rdb_last_bgsave_status', 'N/A')}")
print(f"  Changes since save: {info.get('rdb_changes_since_last_save', 'N/A')}")

### RDB Pros and Cons

| Pros | Cons |
|---|---|
| Compact single file | Data loss between snapshots |
| Fast restart (just load the file) | Fork can be slow with large datasets |
| Great for backups | Not suitable for zero data loss |
| Minimal performance impact | Child process uses memory temporarily |

---
## 2. AOF (Append Only File)

AOF logs **every write command** to a file. On restart, Redis replays the log to rebuild the dataset.

**How it works:**
1. Every write command (SET, INCR, etc.) is appended to the AOF file
2. On restart, Redis replays all commands from the file

**Think of it like:** Writing down every change in a diary.

### Three fsync Policies

| Policy | Description | Safety | Performance |
|---|---|---|---|
| `always` | Write to disk after every command | Safest (no data loss) | Slowest |
| `everysec` | Write to disk every second | Lose max 1 second | Good balance |
| `no` | Let the OS decide when to write | Least safe | Fastest |

In [ ]:
# Check AOF configuration
aof_enabled = r.config_get('appendonly')
aof_fsync = r.config_get('appendfsync')

print(f"AOF enabled: {aof_enabled}")
print(f"AOF fsync policy: {aof_fsync}")

# AOF persistence info
info = r.info('persistence')
print(f"\nAOF info:")
print(f"  AOF enabled:      {info.get('aof_enabled', 'N/A')}")
print(f"  AOF size:         {info.get('aof_current_size', 'N/A')} bytes")
print(f"  AOF rewrite in progress: {info.get('aof_rewrite_in_progress', 'N/A')}")

### AOF Rewrite

Over time, the AOF file grows because it logs every command. Redis can **rewrite** it to be more compact.

Example: If you ran `INCR counter` 1000 times, the AOF has 1000 lines. After rewrite, it becomes just `SET counter 1000`.

In [ ]:
# Trigger a manual AOF rewrite
# Redis CLI: BGREWRITEAOF
try:
    r.bgrewriteaof()
    print("AOF rewrite triggered!")
except redis.ResponseError as e:
    print(f"AOF rewrite note: {e}")
    print("(This is normal if AOF is disabled or a rewrite is already running)")

---
## 3. RDB vs AOF Comparison

| Feature | RDB | AOF |
|---|---|---|
| **Data safety** | May lose minutes of data | Lose at most 1 second |
| **File size** | Compact (compressed) | Larger (all commands logged) |
| **Restart speed** | Fast (load binary) | Slower (replay commands) |
| **Write performance** | No impact between saves | Slight overhead (logging) |
| **Use case** | Backups, disaster recovery | Maximum durability |

### Common Production Setup
Use **both RDB + AOF** for maximum safety:
- AOF for durability (lose at most 1 second)
- RDB for fast backups and disaster recovery

---
## 4. Reading Configuration

In [ ]:
# Get all persistence-related configuration
configs = {
    'save': r.config_get('save'),
    'appendonly': r.config_get('appendonly'),
    'appendfsync': r.config_get('appendfsync'),
    'dbfilename': r.config_get('dbfilename'),
    'dir': r.config_get('dir'),
}

print("=== Persistence Configuration ===")
for key, value in configs.items():
    print(f"  {key}: {value}")

---
## 5. No Persistence Mode (Pure Cache)

If you're using Redis purely as a cache (data can be rebuilt from another source), you can **disable persistence entirely** for maximum performance.

In [ ]:
# To disable all persistence (don't run this unless you mean it!):
# r.config_set('save', '')          # Disable RDB
# r.config_set('appendonly', 'no')   # Disable AOF

print("For pure caching, you would disable both RDB and AOF.")
print("This gives maximum performance but NO data survives a restart.")
print("Only do this when Redis is purely a cache backed by another database.")

---
## 6. Persistence INFO Deep Dive

In [ ]:
info = r.info('persistence')

print("=== Full Persistence Info ===")
for key, value in sorted(info.items()):
    if value != 0 and value != '' and value != -1:
        print(f"  {key}: {value}")

---
## 7. Choosing Persistence for Your Use Case

| Use Case | Recommendation | Why |
|---|---|---|
| **Pure cache** | No persistence | Data can be rebuilt |
| **Session store** | AOF (everysec) | Sessions are important but not critical |
| **Primary database** | RDB + AOF | Maximum durability |
| **Message queue** | AOF (everysec) | Don't want to lose messages |
| **Leaderboard/Stats** | RDB only | Can tolerate some data loss |
| **Job queue** | AOF (everysec) | Jobs shouldn't be lost |

---
## Key Takeaways

| Concept | Details |
|---|---|
| **RDB** | Point-in-time snapshots, compact file, fast restart |
| **AOF** | Logs every write, more durable, larger files |
| **BGSAVE** | Trigger manual RDB snapshot |
| **BGREWRITEAOF** | Compact the AOF file |
| **appendfsync** | always / everysec / no |
| **Production** | Use RDB + AOF together |
| **Pure cache** | Disable both for max performance |

```
BGSAVE                         → Manual RDB snapshot
LASTSAVE                       → When was last save
BGREWRITEAOF                   → Compact AOF file
CONFIG GET save                → Check RDB intervals
CONFIG GET appendonly          → Check if AOF is on
CONFIG GET appendfsync         → Check AOF sync policy
INFO persistence               → Full persistence stats
```

---
## Exercises

1. **Persistence Detective:** Run `INFO persistence` and identify: (a) whether RDB is enabled, (b) whether AOF is enabled, (c) when the last save happened, (d) how many changes have occurred since the last save.

2. **Backup Simulation:** Write 1000 keys, trigger BGSAVE, check LASTSAVE. Write 500 more keys and check `rdb_changes_since_last_save`.

3. **Config Explorer:** List all persistence-related config keys (use `r.config_get('*')` and filter for save/aof/rdb related keys).

4. **Research:** Look up Redis persistence documentation and answer: What happens during a BGSAVE if the server crashes halfway through? (Hint: it uses a temp file)

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 10 — Caching Patterns](./10_Caching_Patterns.ipynb)** — The #1 use case for Redis! Cache-aside, write-through, eviction policies, and more!